In [1]:
from pathlib import Path
import xarray as xr
import itertools as it
import numpy as np
import pandas as pd

p_pro = Path('../../../data/processed/hplc_world/')
p_dat = Path('../../../data/datasets/hplc_world/')
p_dat.mkdir(parents=True, exist_ok=True)

input_vars_legacy_5 = ['412', '442', '490', '560', '673']
input_vars_OLCI_13 = ["400", "412", "442", "490", "510", "560", "620", "665", "673", "681", "708", "778", "865"]
input_vars_OLCI_11 = ["400", "412", "442", "490", "510", "560", "620", "665", "673", "681", "708"]

pigment_names = ['chlide_a[mg*m^3]', 'chla[mg*m^3]', 'chlb[mg*m^3]', 'chlc1+c2[mg*m^3]',
       'fucox[mg*m^3]', "19'hxfcx[mg*m^3]", "19'btfcx[mg*m^3]",
       'diadino[mg*m^3]', 'allox[mg*m^3]', 'diatox[mg*m^3]', 'zeaxan[mg*m^3]',
       'beta_car[mg*m^3]', 'peridinin[mg*m^3]']

### Functions to discard matchups with too much NaNs

In [2]:
# Select data with enough values (discard when majority is nan)

def dimension_len(ds, dim_name):
    if dim_name not in ds.sizes.keys():
        return 1
    return ds.sizes[dim_name]


def select_valid_data(ds, threshold=0.2):
    cube_dimension = (dimension_len(ds, 'lat') * dimension_len(ds, 'lon') *
                      dimension_len(ds, 'time'))
    values_count = (~ds.isnull()).sum(dim=('lat', 'lon'))
    if 'time' in ds.dims:
        values_count = (~ds.isnull()).sum(dim=('time', 'lat', 'lon'))
    no_null_percent = values_count / cube_dimension
    valid_indices = (no_null_percent > threshold).to_array().prod(axis=0) != 0
    return ds.isel(Id=valid_indices)

### Functions to perfrom averages

In [3]:
from src.data.matchups import radius_weighted_average, average


### Load data

In [4]:
rrs_multi_med = xr.load_dataset(p_pro / 'matchups_rrs_multi_med.nc')
# rrs_OLCI_med = xr.load_dataset(p_pro / 'matchups_rrs_OLCI_med.nc')
# rrs_multi_bs = xr.load_dataset(p_pro / 'matchups_rrs_multi_bs.nc')
# rrs_OLCI_bs = xr.load_dataset(p_pro / 'matchups_rrs_OLCI_bs.nc')
rrs_multi = xr.load_dataset(p_pro / 'matchups_rrs_multi.nc')
rrs_OLCI = xr.load_dataset(p_pro / 'matchups_rrs_OLCI.nc')

hplc = xr.load_dataset(p_pro / 'hplc.nc' )
# hplc_train = xr.load_dataset(p_pro / '../dataset_hplc_multi/y.nc')

In [5]:
print("Mediterranean multi:", len(rrs_multi_med.Id))
# print("Mediterranean OLCI:", len(rrs_OLCI_med.Id))

print("World multi:", len(rrs_multi.Id))
print("World OLCI:", len(rrs_OLCI.Id))

Mediterranean multi: 1927
World multi: 32300
World OLCI: 1284


## RRS preprocess

#### Discard matchups containing too much nans

In [6]:
rrs_multi_final   = select_valid_data(rrs_multi, 0.2)
rrs_OLCI_final = select_valid_data(rrs_OLCI, 0.2)
rrs_multi_med_final   = select_valid_data(rrs_multi_med, 0.2)
# rrs_OLCI_med_final = select_valid_data(rrs_OLCI_med, 0.2)

In [7]:
print("Mediterranean multi:", len(rrs_multi_med_final.Id))
# print("Mediterranean OLCI:", len(rrs_OLCI_med_final.Id))

print("World multi:", len(rrs_multi_final.Id))
print("World OLCI:", len(rrs_OLCI_final.Id))

Mediterranean multi: 1493
World multi: 16216
World OLCI: 974


In [8]:

quant = rrs_multi_final[input_vars_legacy_5].where(rrs_multi_final[input_vars_legacy_5] > 0, np.nan).quantile(0.01)
rrs_multi_final[input_vars_legacy_5] = np.maximum(rrs_multi_final[input_vars_legacy_5], quant).drop_vars('quantile')


quant = rrs_OLCI_final[input_vars_OLCI_11].where(rrs_OLCI_final[input_vars_OLCI_11] > 0, np.nan).quantile(0.01)
rrs_OLCI_final[input_vars_OLCI_11] = np.maximum(rrs_OLCI_final[input_vars_OLCI_11], quant).drop_vars('quantile')


quant = rrs_multi_med_final[input_vars_legacy_5].where(rrs_multi_med_final[input_vars_legacy_5] > 0, np.nan).quantile(0.01)
rrs_multi_med_final[input_vars_legacy_5] = np.maximum(rrs_multi_med_final[input_vars_legacy_5], quant).drop_vars('quantile')


# quant = rrs_OLCI_med_final[input_vars_OLCI_13].where(rrs_OLCI_med_final[input_vars_OLCI_13] > 0, np.nan).quantile(0.01)
# rrs_OLCI_med_final[input_vars_OLCI_13] = np.maximum(rrs_OLCI_med_final[input_vars_OLCI_13], quant).drop_vars('quantile')

#### Weighted average

In [9]:
rrs_multi_final = radius_weighted_average(np.log(rrs_multi_final))

In [10]:
rrs_OLCI_final = radius_weighted_average(np.log(rrs_OLCI_final))

In [11]:
rrs_multi_med_final = radius_weighted_average(np.log(rrs_multi_med_final))

In [12]:
# rrs_OLCI_med_final = radius_weighted_average(np.log(rrs_OLCI_med_final))

#### Pigments preprocess

In [13]:
eps = 0.001


In [14]:
# Limit the pigment values to a reasonable range (0 -- 5)
hplc[pigment_names] = hplc[pigment_names].where(hplc[pigment_names] >= 0, np.nan)
hplc[pigment_names] = hplc[pigment_names].clip(0, 5)


In [15]:
# Apply logarithms
quant = hplc[pigment_names].where(hplc[pigment_names] > eps, np.nan).quantile(0.05)
hplc[pigment_names] = np.maximum(hplc[pigment_names], quant).drop_vars('quantile')
hplc[pigment_names] = np.log(hplc[pigment_names])

# Limit depth to 5 metters
depth_index = (hplc.depth.values <= 5) + (hplc.depth.isnull().values)
hplc_final = hplc.sel(Id=depth_index)

In [16]:
hplc_final[pigment_names]

<xarray.Dataset>
Dimensions:            (Id: 17312)
Coordinates:
  * Id                 (Id) int32 0 1 2 3 4 5 ... 53884 53886 53888 53890 53893
Data variables: (12/13)
    chlide_a[mg*m^3]   (Id) float64 -4.51 -4.51 -3.507 ... -5.521 -4.962 -3.772
    chla[mg*m^3]       (Id) float64 -0.3827 -0.3725 0.1773 ... -1.115 -0.1567
    chlb[mg*m^3]       (Id) float64 -3.17 -3.474 -3.54 ... -2.83 -3.037 -2.765
    chlc1+c2[mg*m^3]   (Id) float64 -3.219 -3.27 -2.551 ... -3.65 -3.244 -2.163
    fucox[mg*m^3]      (Id) float64 -1.808 -1.79 -1.194 ... -3.411 -2.703 -1.378
    19'hxfcx[mg*m^3]   (Id) float64 -5.098 -5.098 -5.098 ... -2.797 -1.981
    ...                 ...
    diadino[mg*m^3]    (Id) float64 -3.612 -3.612 -2.957 ... -3.101 -2.577
    allox[mg*m^3]      (Id) float64 -3.507 -3.411 -2.375 ... -6.32 -6.215 -3.912
    diatox[mg*m^3]     (Id) float64 -6.387 -6.387 -6.387 ... -4.828 -3.772
    zeaxan[mg*m^3]     (Id) float64 -4.51 -4.605 -4.423 ... -3.324 -3.772 -4.269
    beta_car[mg*m^3]   (Id) float64 nan nan nan nan nan ... nan nan nan nan nan
    peridinin[mg*m^3]  (Id) float64 -4.017 -3.912 -3.381 ... -4.269 -2.765

### Match rrs and hplc

In [17]:
index_multi   = np.intersect1d(hplc_final.Id.values, rrs_multi_final.Id.values)
index_OLCI   = np.intersect1d(hplc_final.Id.values, rrs_OLCI_final.Id.values)
index_multi_med   = np.intersect1d(hplc_final.Id.values, rrs_multi_med_final.Id.values)
# index_OLCI_med   = np.intersect1d(hplc_final.Id.values, rrs_OLCI_med_final.Id.values)

### Mediterranean
hplc_multi_med_final = hplc_final.sel(Id=index_multi_med)
rrs_multi_med_final  = rrs_multi_med_final.sel(Id=index_multi_med)

# hplc_OLCI_med_final = hplc_final.sel(Id=index_OLCI_med)
# rrs_OLCI_med_final  = rrs_OLCI_med_final.sel(Id=index_OLCI_med)

### World
hplc_multi_final = hplc_final.sel(Id=index_multi)
rrs_multi_final  = rrs_multi_final.sel(Id=index_multi)

hplc_OLCI_final = hplc_final.sel(Id=index_OLCI)
rrs_OLCI_final  = rrs_OLCI_final.sel(Id=index_OLCI)

In [18]:
print("Mediterranean multi:", len(rrs_multi_med_final.Id))
# print("Mediterranean OLCI:", len(rrs_OLCI_med_final.Id))

print("World multi:", len(rrs_multi_final.Id))
print("World OLCI:", len(rrs_OLCI_final.Id))

Mediterranean multi: 86
World multi: 7287
World OLCI: 668


In [19]:
### Mediterranean
hplc_multi_med_final.to_dataframe().to_csv(p_dat / 'hplc_multi_med.csv')
rrs_multi_med_final.to_dataframe().to_csv(p_dat / 'rrs_multi_med.csv')

# hplc_OLCI_med_final.to_dataframe().to_csv(p_dat / 'hplc_OLCI_med.csv')
# rrs_OLCI_med_final.to_dataframe().to_csv(p_dat / 'rrs_OLCI_med.csv')

### World
hplc_multi_final.to_dataframe().to_csv(p_dat / 'hplc_multi.csv')
rrs_multi_final.to_dataframe().to_csv(p_dat / 'rrs_multi.csv')

hplc_OLCI_final.to_dataframe().to_csv(p_dat / 'hplc_OLCI.csv')
rrs_OLCI_final.to_dataframe().to_csv(p_dat / 'rrs_OLCI.csv')

In [20]:
hplc_multi_final

<xarray.Dataset>
Dimensions:                                              (Id: 7287)
Coordinates:
  * Id                                                   (Id) int32 171 ... 5...
Data variables: (12/354)
    received                                             (Id) <U66 '20140912....
    identifier_product_doi                               (Id) <U39 '10.5067_S...
    investigators                                        (Id) <U38 'Collin_Ro...
    affiliations                                         (Id) <U15 'Bowdoin_C...
    contact                                              (Id) <U41 'croesler@...
    experiment                                           (Id) <U11 'ThreeRive...
    ...                                                   ...
    tpg_20filt_bincount                                  (Id) float64 nan ......
    dp_20filt                                            (Id) float64 nan ......
    dp_20filt_bincount                                   (Id) float64 nan ......
    number_of_data_rows                                  (Id) float64 nan ......
    instrument_manufacturer                              (Id) <U7 'nan' ... '...
    instrument_model                                     (Id) <U6 'nan' ... '...

In [21]:
set(hplc_multi_final['identifier_product_doi'].values)

{'10.5067_SeaBASS_ACIDD_DATA001',
 '10.5067_SeaBASS_AMLR_DATA001',
 '10.5067_SeaBASS_BENTHICECOLOGY_FROMSPAC',
 '10.5067_SeaBASS_BIOCOMPLEXITY_DATA001',
 '10.5067_SeaBASS_BOWDOINBUOY_DATA001',
 '10.5067_SeaBASS_CASES_DATA001',
 '10.5067_SeaBASS_CCE-LTER_DATA001',
 '10.5067_SeaBASS_CHESAPEAKE_LIGHT_TOWER_',
 '10.5067_SeaBASS_ECOHAB_DATA001',
 '10.5067_SeaBASS_ECOMON_DATA001',
 '10.5067_SeaBASS_EGEE3_DATA001',
 '10.5067_SeaBASS_EGEE5_DATA001',
 '10.5067_SeaBASS_GOMECC_DATA001',
 '10.5067_SeaBASS_GULFCARBON_DATA001',
 '10.5067_SeaBASS_INTRO_DATA001',
 '10.5067_SeaBASS_LAMONT_SCS_DATA001',
 '10.5067_SeaBASS_LATTE_DATA001',
 '10.5067_SeaBASS_LMER-TIES_DATA001',
 '10.5067_SeaBASS_MOCE_PIGMENT_DATABASE_D',
 '10.5067_SeaBASS_MURI_CAMOUFLAGE_DATA001',
 '10.5067_SeaBASS_MVCO_DATA001',
 '10.5067_SeaBASS_NES-LTER_DATA001',
 '10.5067_SeaBASS_NOAA_DATA001',
 '10.5067_SeaBASS_NSF-BWZ_DATA001',
 '10.5067_SeaBASS_NSF_GULF_RAPID_DATA001',
 '10.5067_SeaBASS_PACE_ABSCLOSURE_DATA001',
 '10.5067_SeaBASS_SAB

In [22]:
set(hplc_OLCI_final['identifier_product_doi'].values)

{'10.5067_SeaBASS_ACIDD_DATA001',
 '10.5067_SeaBASS_BOWDOINBUOY_DATA001',
 '10.5067_SeaBASS_ECOMON_DATA001',
 '10.5067_SeaBASS_INTRO_DATA001',
 '10.5067_SeaBASS_LAMONT_SCS_DATA001',
 '10.5067_SeaBASS_NES-LTER_DATA001',
 '10.5067_SeaBASS_NSF_GULF_RAPID_DATA001',
 '10.5067_SeaBASS_PACE_ABSCLOSURE_DATA001',
 '10.5067_SeaBASS_SFMBON_DATA001',
 'nan'}

In [23]:
#  World restructed to EUR:

# multi
lat_eur = (rrs_multi_final.lat > 28) * (rrs_multi_final.lat < 75)
lon_eur = (rrs_multi_final.lon > -10) * (rrs_multi_final.lon < 43)

indx_eur = lat_eur * lon_eur
print(indx_eur.sum())

hplc_multi_EU_final = hplc_multi_final.sel(Id=indx_eur)
rrs_multi_EU_final = rrs_multi_final.sel(Id=indx_eur)

hplc_multi_EU_final.to_dataframe().to_csv(p_dat / 'hplc_multi_EU.csv')
rrs_multi_EU_final.to_dataframe().to_csv(p_dat / 'rrs_multi_EU.csv')


lat_eur = (rrs_OLCI_final.lat > 28) * (rrs_OLCI_final.lat < 75)
lon_eur = (rrs_OLCI_final.lon > -10) * (rrs_OLCI_final.lon < 43)

indx_eur = lat_eur * lon_eur
print(indx_eur.sum())

hplc_OLCI_EU_final = hplc_OLCI_final.sel(Id=indx_eur)
rrs_OLCI_EU_final = rrs_OLCI_final.sel(Id=indx_eur)

hplc_OLCI_EU_final.to_dataframe().to_csv(p_dat / 'hplc_OLCI_EU.csv')
rrs_OLCI_EU_final.to_dataframe().to_csv(p_dat / 'rrs_OLCI_EU.csv')


<xarray.DataArray ()>
array(88)
<xarray.DataArray ()>
array(0)


In [24]:
dat = pd.read_csv(p_dat / 'hplc_OLCI.csv')[["chlide_a[mg*m^3]", "chla[mg*m^3]", "chlb[mg*m^3]", "chlc1+c2[mg*m^3]",
    "fucox[mg*m^3]", "19'hxfcx[mg*m^3]", "19'btfcx[mg*m^3]",
    "diadino[mg*m^3]", "allox[mg*m^3]", "diatox[mg*m^3]", "zeaxan[mg*m^3]",
    "beta_car[mg*m^3]", "peridinin[mg*m^3]"]]

In [25]:
np.exp(rrs_OLCI_final.mean())

<xarray.Dataset>
Dimensions:  ()
Data variables: (12/13)
    400      float64 0.002329
    412      float64 0.002684
    442      float64 0.003973
    490      float64 0.005872
    510      float64 0.005801
    560      float64 0.005298
    ...       ...
    665      float64 0.0006072
    673      float64 0.0005868
    681      float64 0.0005887
    708      float64 0.0002054
    lat      float32 7.949e+13
    lon      float32 5.902e-34

In [26]:
dat.isna().sum()

chlide_a[mg*m^3]      64
chla[mg*m^3]           3
chlb[mg*m^3]          54
chlc1+c2[mg*m^3]      55
fucox[mg*m^3]          2
19'hxfcx[mg*m^3]      29
19'btfcx[mg*m^3]     254
diadino[mg*m^3]        2
allox[mg*m^3]         94
diatox[mg*m^3]        39
zeaxan[mg*m^3]        19
beta_car[mg*m^3]     668
peridinin[mg*m^3]     11
dtype: int64

In [27]:
dat.mean(axis=0)

chlide_a[mg*m^3]    -3.624931
chla[mg*m^3]        -0.167898
chlb[mg*m^3]        -2.748993
chlc1+c2[mg*m^3]    -2.410899
fucox[mg*m^3]       -1.825585
19'hxfcx[mg*m^3]    -3.282180
19'btfcx[mg*m^3]    -4.346455
diadino[mg*m^3]     -2.973725
allox[mg*m^3]       -3.851994
diatox[mg*m^3]      -4.939881
zeaxan[mg*m^3]      -2.809514
beta_car[mg*m^3]          NaN
peridinin[mg*m^3]   -3.971908
dtype: float64

In [28]:
import pickle
with open('../../../experiments/OLCI_sat/split_0/data/test_ids', "rb") as f:
     test = pickle.load(f)
with open('../../../experiments/OLCI_sat/split_0/data/train_ids', "rb") as f:
     train = pickle.load(f)

In [29]:
test

array([665,  40, 482,  53, 661, 303, 221, 406, 162, 233, 349,  24, 195,
       489, 496, 642,  78, 554, 159, 320, 667,  76, 573,  71, 180, 582,
       109, 115, 385, 427, 326,  13, 300, 411, 570, 361, 157, 651,   4,
        99, 430, 626, 434, 241, 203, 515,  81,  12, 237, 271, 222, 465,
       225, 396, 213, 379, 649, 154, 364,  43,  41, 450, 585, 166, 332,
       578, 462,  84, 260, 540, 597, 467, 549, 324, 278, 599,  82, 518,
       281,  57, 360,  52, 506,  75, 200, 442, 454, 559, 455, 497, 113,
        27, 534, 631, 302, 313, 327, 354, 580, 119, 140], dtype=int64)